<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/NormalGPTPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
algozee_ai_generated_vs_human_written_text_dataset_path = kagglehub.dataset_download('algozee/ai-generated-vs-human-written-text-dataset')

print('Data source import complete.')


In [ ]:
import random
import numpy as np
import torch

def set_Seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

set_Seed(42)

In [ ]:
!pip install transformers

In [ ]:
!pip install -q datasets

In [ ]:
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel,AutoTokenizer
import pandas as pd
import os
from google.colab import userdata
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,Dataset
from tqdm import tqdm
import gc

In [ ]:
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# !kaggle datasets download -d algozee/ai-generated-vs-human-written-text-dataset

In [ ]:
# !unzip -q ai-generated-vs-human-written-text-dataset.zip -d ./appstore/

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/algozee/ai-generated-vs-human-written-text-dataset/AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv')
df.head(5)

,text_id,content_text,author_type,model_source,prompt_complexity_score,perplexity_score,burstiness_index,syntactic_variability,semantic_coherence_score,lexical_diversity_ratio,readability_grade_level,generation_confidence_score
0,TXT_0001,learning pattern detection algorithm pattern n...,AI,Human,0.029,73.75,0.953,0.465,0.351,0.187,12.2,0.162
1,TXT_0002,algorithm algorithm data research network mode...,Human,Claude,0.605,43.11,0.054,0.952,0.314,0.636,9.8,0.012
2,TXT_0003,analysis language generation research pattern ...,Human,GPT-4,0.396,59.97,0.709,0.945,0.684,0.500,13.5,0.171
3,TXT_0004,data language system learning content data net...,AI,GPT-4,0.299,18.99,0.532,0.780,0.216,0.103,12.9,0.838
4,TXT_0005,model learning content language model generati...,AI,Human,0.867,82.45,0.478,0.602,0.420,0.198,6.4,0.022


In [ ]:
df.isnull().sum()

text_id                        0
content_text                   0
author_type                    0
model_source                   0
prompt_complexity_score        0
perplexity_score               0
burstiness_index               0
syntactic_variability          0
semantic_coherence_score       0
lexical_diversity_ratio        0
readability_grade_level        0
generation_confidence_score    0
dtype: int64

In [ ]:
df = df[df['content_text'].str.strip() != ""]

In [ ]:
statement_text = df['content_text'].astype(str).values

In [ ]:
x_train,x_test = train_test_split(statement_text,test_size=0.2,random_state=42)

In [ ]:
autoToken = AutoTokenizer.from_pretrained('gpt2')
autoToken.pad_token = autoToken.eos_token

In [ ]:
c = df['content_text'].str.split().str.len().mean()
c

np.float64(11.015)

In [ ]:
max_len = 64
vocab_size = autoToken.vocab_size

In [ ]:
train_token = autoToken(text=list(x_train),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
test_token = autoToken(text=list(x_test),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class NormalGPT(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids[:-1],
        'attention_mask':attention_mask[:-1],
        'target_ids':input_ids[1:]
    }


In [ ]:
train_loader = DataLoader(dataset=NormalGPT(train_token),batch_size=32,shuffle=True,pin_memory=True,num_workers=2)
val_loader = DataLoader(dataset=NormalGPT(test_token),batch_size=32,shuffle=True,pin_memory=True,num_workers=2)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
class GPTWritter(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.gpt2 = AutoModel.from_pretrained('gpt2')
    self.dropout = nn.Dropout(0.2)
    self.outputlayer = nn.Linear(768,vocab_size)

  def forward(self,input_ids,mask):
    output = self.gpt2(input_ids=input_ids,attention_mask=mask)
    x = output.last_hidden_state
    x = self.dropout(x)
    return self.outputlayer(x)

In [ ]:
model = GPTWritter(vocab_size=vocab_size)
if torch.cuda.device_count() > 1:
  model = nn.DataParallel(model)
model = model.to(device)

In [ ]:
optimizer = optim.AdamW(model.parameters(),lr = 5e-5)
loss_fn = nn.CrossEntropyLoss()
scalar = torch.cuda.amp.GradScaler()

/tmp/ipykernel_57/2410347667.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scalar = torch.cuda.amp.GradScaler()


In [ ]:
epochs = 4

for epoch in range(epochs):
  #________Train___________#
  model.train()
  train_loss = 0
  progressbar_train = tqdm(train_loader,desc=f'Epoch: {epoch+1}')

  for batch in progressbar_train:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    target_ids = batch['target_ids'].to(device)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
      outputs = model(input_ids,attention_mask)
      loss = loss_fn(outputs.view(-1,outputs.size(-1)),target_ids.view(-1))

    scalar.scale(loss).backward()
    scalar.step(optimizer=optimizer)
    scalar.update()

    train_loss += loss.item()
    progressbar_train.set_postfix(loss=loss.item())

  average_loss = train_loss / len(train_loader)
  train_perplexity = torch.exp(torch.tensor(average_loss))

  print(f'Train_loss: {average_loss:.4f} | Train_per: {train_perplexity:.2f}')

#________Validation___________#
  model.eval()
  val_loss = 0
  progressbar_val = tqdm(val_loader,desc=f'Validation: ')

  with torch.no_grad():
   for val_batch in progressbar_val:
    val_input_ids = val_batch['input_ids'].to(device)
    val_attention_mask = val_batch['attention_mask'].to(device)
    val_target_ids = val_batch['target_ids'].to(device)

    with torch.cuda.amp.autocast():
      val_outputs = model(val_input_ids,val_attention_mask)
      val_loss = loss_fn(val_outputs.view(-1,val_outputs.size(-1)),val_target_ids.view(-1))

    val_loss += val_loss.item()
    progressbar_val.set_postfix(loss=val_loss.item())

  val_average_loss = val_loss / len(val_loader)
  val_perplexity = torch.exp(torch.tensor(val_average_loss))

  print(f'Val_loss: {val_average_loss:.4f} | Val_per: {val_perplexity:.2f}')

  gc.collect()
  torch.cuda.empty_cache()

Epoch: 1:   0%|          | 0/25 [00:00<?, ?it/s]/tmp/ipykernel_57/1281486591.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch: 1: 100%|██████████| 25/25 [00:07<00:00,  3.30it/s, loss=1.15]


Train_loss: 1.6572 | Train_per: 5.24


Validation:   0%|          | 0/7 [00:00<?, ?it/s]/tmp/ipykernel_57/1281486591.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validation: 100%|██████████| 7/7 [00:00<00:00,  8.39it/s, loss=2.05]
/tmp/ipykernel_57/1281486591.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_perplexity = torch.exp(torch.tensor(val_average_loss))


Val_loss: 0.2931 | Val_per: 1.34


Epoch: 2: 100%|██████████| 25/25 [00:07<00:00,  3.29it/s, loss=0.751]


...

Val_loss: 0.1584 | Val_per: 1.17


Epoch: 4: 100%|██████████| 25/25 [00:07<00:00,  3.30it/s, loss=0.611]


Train_loss: 0.5838 | Train_per: 1.79


Validation: 100%|██████████| 7/7 [00:00<00:00,  8.35it/s, loss=1.05]


Val_loss: 0.1494 | Val_per: 1.16


In [ ]:
def generate_news(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()

    # 1. Turn your headline prompt into token numbers
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Extract the true inner model if DataParallel is active
    actual_model = model.module if isinstance(model, nn.DataParallel) else model

    for _ in range(max_new_tokens):
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                # 2. Get the vocabulary predictions matrix
                logits = actual_model(input_ids, mask=None)

        # 3. Focus entirely on the predictions for the VERY LAST word slot
        next_token_logits = logits[:, -1, :]

        # 4. Select the highest scoring word (Classification winner!)
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        # 5. Concatenate the new word token onto the sequence string array
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # Stop typing if the model hits the End-of-Text boundary token
        if next_token.item() == tokenizer.eos_token_id:
            break

    # 6. Translate the raw numbers back into readable English text
    return tokenizer.decode(input_ids, skip_special_tokens=True)[0]

# 💥 RUN THESE EXAMPLES IN A NEW CELL TO SEE IT WRITE
print("--- TEST 1 ---")
my_coding_prompt = "Instruction: Write a Python function to check if a number is even.\n### Code:\n"
print(generate_news(model, autoToken, my_coding_prompt))